In [8]:
%pip install earthengine-api pandas numpy matplotlib seaborn geopandas folium

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import ee

In [6]:
ee.Authenticate()


Successfully saved authorization token.


In [7]:
ee.Initialize(project='project-0cf410a3-f35f-4911-9c5')

### Definitions
- What is CHRIPS?
    - Climate Hazards Group InfraRed Precipitation with Station data
    - It's a satellite derived daily rainfall dataset, across Africa and other regions.
- Spatial resolution: 0.05 degrees, meaning that each data point covers a 5km of estimation of rain fall.
- Temporal resolution(cadence): Daily (how data is updated)
- Period: 1981 - near present (2/3 weeks delay)


In [15]:
import ee
import datetime

SAFARI_BBOX = ee.Geometry.BBox(34.00, -4.50, 37.00, -1.50)

In [18]:
# total available data

no_filtered_chirps = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
          .filterBounds(SAFARI_BBOX))

dates = no_filtered_chirps.aggregate_array('system:time_start').getInfo()

readable_dates = [
    datetime.datetime.fromtimestamp(d/1000).strftime('%Y-%m-%d')
    for d in dates
]

print(f'Date range: {readable_dates[0]} to {readable_dates[-1]}')
print(f'Days with data: {len(readable_dates)}')

Date range: 1981-01-01 to 2026-05-31
Days with data: 16587


In [34]:
#Filtering data

DATE_START = '2023-11-01'
DATE_END   = '2023-11-30'

filtering_chirps = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
          .filterDate(DATE_START, DATE_END)
          .filterBounds(SAFARI_BBOX))

In [35]:
# Available data for the given date


dates = filtering_chirps.aggregate_array('system:time_start').getInfo()
readable_dates = [
    datetime.datetime.fromtimestamp(d/1000).strftime('%Y-%m-%d')
    for d in dates
]
print(f'Date range: {readable_dates[0]} to {readable_dates[-1]}')
print(f'Days with data: {len(readable_dates)}')

Date range: 2023-11-01 to 2023-11-29
Days with data: 29


In [36]:
# There's a missing day...
import pandas as pd

expected_dates = pd.date_range(
    start=DATE_START,
    end=DATE_END,
    freq='D'
).strftime('%Y-%m-%d').tolist()

missing = set(expected_dates) - set(readable_dates)

for d in sorted(missing):
    print(f"{d}")

  2023-11-30
